In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
import os, pathlib, glob, random
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.metrics import confusion_matrix
from transformers import WhisperProcessor, WhisperForConditionalGeneration
# from datasets import load_dataset
from transformers.models.whisper.modeling_whisper import WhisperModel, WhisperEncoder
from transformers.models.whisper.configuration_whisper import WhisperConfig
from typing import Optional, Tuple, Union
import torch
import librosa 
import matplotlib.pyplot as plt
import numpy as np
import os, glob, pickle
import scipy.io as sio
from tqdm import tqdm
import multiprocessing as mp 
import torch.optim as optim

In [ ]:
# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'


In [ ]:
#checking for gpu 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
print(torch.__version__)
# print(mp.cpu_count())


In [ ]:
batch_size = 64
# learning_rate = 0.01
# learning_rate = 0.03
# num_epochs = 40

In [ ]:
train_data_path = r"D:\mahesh_feature\static\10_static\training"
valid_data_path = r"D:\mahesh_feature\static\10_static\validation"
test_data_path = r"D:\mahesh_feature\static\10_static\testing"

In [ ]:
# drop_amount = 0.2

# class WhisperWordClassifier(nn.Module):

#   def __init__(self):
#         super().__init__()
  
#         self.hidden_size = hidden_size
#         self.lstm1 = nn.LSTM(768, 128, bidirectional=True, batch_first=True)
#         self.dropout1 = nn.Dropout(drop_amount)
#         self.lstm2 = nn.LSTM(768, 128, bidirectional=True, batch_first=True)
#         self.dropout2 = nn.Dropout(drop_amount)
#         self.fc = nn.Linear(hidden_size*2, 155)
#         self.softmax = nn.Softmax(dim=1)

#   def forward(self, out):

#         lstm_out, _ = self.lstm1(out)
#         out = self.dropout1(lstm_out)
#         lstm_out, _ = self.lstm1(out)
#         out = self.dropout1(lstm_out)
#         out = self.fc(out)
#         out = self.softmax(out)
#         return out
# # 

In [ ]:
# drop_amount = 0.255

# class LSTMClassifier(nn.Module):
#     def __init__(self, input_size, hidden_size, num_layers, num_classes):
#         super(LSTMClassifier, self).__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers
#         self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=False)
#         self.dropout = nn.Dropout(p=drop_amount)
#         self.fc = nn.Linear(hidden_size, num_classes)

#     def forward(self, x):
#         h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device=x.device)
#         c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device=x.device)
#         out, _ = self.lstm(x, (h0, c0))
#         out = self.dropout(out)
#         out = self.fc(out[:, -1, :])
#         return out

In [ ]:
# drop_amount = 0.255

# class BiLSTMClassifier(nn.Module):
#     def __init__(self, input_size, hidden_size, num_layers, num_classes):
#         super(BiLSTMClassifier, self).__init__()
#         self.hidden_size = hidden_size
#         self.num_layers = num_layers
#         self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
#         self.dropout = nn.Dropout(p=drop_amount)
#         self.fc = nn.Linear(hidden_size*2, num_classes)

#     def forward(self, x):
#         h0 = torch.zeros(self.num_layers*2, x.size(0), self.hidden_size).to(device=x.device)
#         c0 = torch.zeros(self.num_layers*2, x.size(0), self.hidden_size).to(device=x.device)
#         out, _ = self.lstm(x, (h0, c0))
#         out = self.dropout(out)
#         # Extract the output of the last time step from both directions
#         last_hidden_state = torch.cat((out[:, -1, :self.hidden_size], out[:, 0, self.hidden_size:]), dim=1)
#         output = self.fc(last_hidden_state)
#         return output

In [ ]:
# drop_amount = 0.255

# class BiGRUAudioClassifier(nn.Module):
#     def __init__(self,input_size, num_classes, hidden_units, num_layers):
#         super(BiGRUAudioClassifier, self).__init__()
#         self.input_size = input_size 
#         self.num_classes = num_classes
#         self.hidden_units = hidden_units
#         self.num_layers = num_layers

#         self.bigru = nn.GRU(input_size=input_size, hidden_size=hidden_units, num_layers=num_layers, batch_first=True, bidirectional=False)
#         self.dropout = nn.Dropout(p=drop_amount)
#         # self.fc = nn.Linear(hidden_units, num_classes)
#         self.fc = nn.Linear(hidden_units * 2, num_classes)

#     def forward(self, x):
#         # x: (batch_size, sequence_length, num_features)

#         # Pass the input through the bi-GRU layers
#         output, _ = self.bigru(x)
#         output = self.dropout(output)
#         # Extract the last hidden state (concatenate forward and backward hidden states)
#         last_hidden_state = torch.cat((output[:, -1, :self.hidden_units], output[:, 0, self.hidden_units:]), dim=1)
        
#         # Apply the fully connected layer for classification
#         output = self.fc(last_hidden_state)

#         return output

In [ ]:
# del model_whisp
torch.cuda.empty_cache()

In [ ]:
# model_whisp = CapsuleNet()
# model_whisp.to(device)

OLD CODE MODEL DEFINED HERE:

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn.datasets import load_breast_cancer
# from sklearn.model_selection import train_test_split
# from sklearn.svm import SVC
# from sklearn.metrics import roc_curve

# # Step 1: Load the dataset
# # data = load_breast_cancer()
# # X = data.data
# # y = data.target
# import numpy as np

# # Generate a large dataset
# num_samples = 1000  # Number of samples
# num_features = 10  # Number of features

# # Generate random features
# X = np.random.rand(num_samples, num_features)

# # Generate random labels (binary classification)
# y = np.random.randint(0, 2, num_samples)

# # Print the shape of the dataset
# print("Shape of X:", X.shape)
# print("Shape of y:", y.shape)


# # Step 2: Split the dataset into train and test sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# print("1")
# # Step 3: Train the model on the training set
# model = SVC(probability=True)  # Using Support Vector Classifier as an example
# model.fit(X_train, y_train)
# print("1")
# # Step 4: Generate prediction scores on the test set
# y_scores = model.predict_proba(X_test)[:, 1]  # Considering binary classification and using class probabilities
# print("1")
# # Step 5: Calculate FAR-FRR curve and EER
# fpr, tpr, thresholds = roc_curve(y_test, y_scores)
# fnr = 1 - tpr
# eer_threshold = thresholds[np.nanargmin(np.abs(fnr - fpr))]
# eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
# print("1")
# # Step 6: Plot the ROC curve
# plt.plot(fpr, tpr)
# plt.xlabel('False Positive Rate (FPR)')
# plt.ylabel('True Positive Rate (TPR)')
# plt.title('Receiver Operating Characteristic (ROC) Curve')
# plt.show()
# print("1")
# # Step 7: Print EER
# print('Equal Error Rate (EER):', eer)


In [ ]:
# # Define the parameters
# input_size = 512
# hidden_size = 256
# num_layers = 2
# num_classes = 2

# # model_whisp= WhisperWordClassifier()
# model_whisp = BiLSTMClassifier(input_size, hidden_size, num_layers, num_classes)
# # model_whisp = BiGRUAudioClassifier(input_size, num_classes, hidden_size, num_layers)
# model_whisp.to(device)

#### Custom Dataset

In [ ]:
# class PtDataset(Dataset):
#     def __init__(self, directory):
#         self.directory = directory
#         self.classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
#         self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
#         self.files = []
#         for c in self.classes:
#             c_dir = os.path.join(directory, c)
#             # c_files = [(f, self.class_to_idx[c]) for f in glob.glob(os.path.join(c_dir, '*.pt'))]
#             c_files = [(os.path.join(c_dir, f), self.class_to_idx[c]) for f in os.listdir(c_dir)]
#             self.files.extend(c_files)
#         random.shuffle(self.files)

#     def __len__(self):
#         return len(self.files)
    
#     def __getitem__(self, idx):
#         filepath, label = self.files[idx]
#         # desired_dimensions = (1, 400, 1024)
        
#         try:
#             data = torch.load(filepath, map_location='cpu').detach()
#             # if(data.shape[1]<400) :
#             #     padding_amounts = (0,0,0,desired_dimensions[1] - data.shape[1])
#             #     data = F.pad(data, padding_amounts, mode='constant',value = 0)
#                 # if(len(data.shape)<3):
#                 #     print(data.shape)
#                 #     print(filepath, " pad")
#             data = data[-1, 0:100 , :]
#                 # if(len(data.shape)<3):
#                 #     print(data.shape)
#                 #     print(filepath, " no pad")
            
#         except Exception as e:
#             print(f"Error loading file {filepath}: {str(e)}")
#             return None
#         return data, label

class PtDataset(Dataset):
    def __init__(self, directory):
        self.directory = directory
        self.classes = sorted(entry.name for entry in os.scandir(directory) if entry.is_dir())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.files = []
        for c in self.classes:
            c_dir = os.path.join(directory, c)
            c_files = [(os.path.join(c_dir, f), self.class_to_idx[c]) for f in os.listdir(c_dir)]
            self.files.extend(c_files)
        random.shuffle(self.files)
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        filepath, label = self.files[idx]
        try:
            mat_vals = scipy.io.loadmat(filepath) 
            data = mat_vals['final']
            data = data.T
            max_len=600
            if (max_len > data.shape[0]):
                pad_width = max_len - data.shape[0]
                data = np.pad(data, pad_width=((0, pad_width),(0,0)), mode='constant')
            else:
                data = data[:max_len, :]
        except Exception as e:
            print(f"Error loading file {filepath}: {str(e)}")
            return None
        return data, label

In [ ]:
train_dataset = PtDataset(train_data_path)
valid_dataset = PtDataset(valid_data_path)
test_dataset = PtDataset(test_data_path)

#### Custom Dataloader

In [ ]:
class PtDataLoader(DataLoader):
    def __init__(self, directory, batch_size, shuffle=True, num_workers=8):
        dataset = PtDataset(directory)
        super().__init__(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)

In [ ]:
train_dataloader = PtDataLoader(directory=train_data_path, batch_size=batch_size)
valid_dataloader = PtDataLoader(directory= valid_data_path,batch_size=batch_size)
test_dataloader = PtDataLoader(directory=test_data_path, batch_size=batch_size)

In [ ]:
train_count = len(train_dataset) 
valid_count = len(valid_dataset)
test_count = len(test_dataset)
print(train_count)
print(valid_count)
print(test_count)

#### TRAINING THE MODEL

In [ ]:
# Defining loss and optimizer for BiLSTM Model
loss_function = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model_whisp.parameters(), lr=learning_rate)

# Defining loss and optimizer for CAPSNET Model
# loss_function = CapsuleLoss()
# optimizer = optim.Adam(model_whisp.parameters())

In [ ]:
#Model training and testing 

n_total_steps = len(train_dataloader) # n_total_steps * batch size will give total number of training files (consider that last batch may not be fully filled)
train_accuracy_list = []
train_loss_list = []
valid_accuracy_list = []

max_acc = 0
pred_labels =[]
act_labels = []
pred =[]
lab =[]

In [ ]:
torch.cuda.empty_cache()

In [ ]:
import torch
import torch.nn as nn
# del model_whisp
# torch.cuda.empty_cache()


class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.cnn2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=4, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)
        self.cnn3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=1)
        self.relu3 = nn.ReLU()
        self.cnn4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=5, stride=1, padding=1)
        self.relu4 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        self.linear1 = nn.Linear(352000, 128)
        self.linear2 = nn.Linear(128, 64)
        self.linear3 = nn.Linear(64, 16)
        self.linear4 = nn.Linear(16, 1)

    def forward(self, x):
        out = self.cnn1(x)
        out = self.relu1(out)
        out = self.cnn2(out)
        out = self.relu2(out)
        out = self.maxpool1(out)

        out = self.cnn3(out)
        out = self.relu3(out)
        out = self.cnn4(out)
        out = self.relu4(out)
        out = self.maxpool2(out)

        out = out.view(out.size(0), -1)

        out = self.linear1(out)
        out = self.linear2(out)
        out = self.linear3(out)

        return out


# # Create an instance of the model
# model_whisp = CNNModel()
# num_epochs = 15
# # optimizer = torch.optim.SGD(model_whisp.parameters(), lr=0.001)
# optimizer = torch.optim.Adam(model_whisp.parameters(), lr=0.001)

In [ ]:
# model = CNNModel()
# print(model)

In [ ]:
# Create an instance of the model
model = CNNModel().to(device)
num_epochs = 15
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
print(model)

In [ ]:
# del model_whisp
# torch.cuda.empty_cache()

In [ ]:
# model_whisp = cnn_model()
# model_whisp = BiGRUAudioClassifier(input_size, num_classes, hidden_size, num_layers)
# model_whisp.to(device)

In [ ]:
# # Create a DataLoader for the dataset
# batch_size = 64
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# # Iterate over the data loader
# for batch_images, batch_labels in train_loader:
#     # Forward pass and loss calculation
#     outputs = model_whisp(batch_images)
#     loss = loss_function(outputs, batch_labels)
#     # Rest of the training loop
#     ...


In [ ]:
#updated code is here..

In [ ]:
# num_epochs = 50

In [ ]:
#old_code
import torch
from torch.autograd import Variable
from tqdm import tqdm

for epoch in range(num_epochs):
    # Evaluation and training on training dataset
    model.train()
    train_accuracy = 0.0
    train_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(tqdm(train_dataloader)):
        if torch.cuda.is_available():
            images = images.to(device)
            labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Reshape the input tensor to have 4 dimensions
        images = images.unsqueeze(1)
        images = images.float()  # Convert the tensor to float if needed
        
        outputs = model(images)
#         print('Outputs shape:', outputs.shape)
#         print('Labels shape:', labels.shape)
        
        loss = loss_function(outputs, labels)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.cpu().data * images.size(0)
        _, prediction = torch.max(outputs.data, 1)
        
        train_accuracy += int(torch.sum(prediction == labels.data))
        
    train_accuracy = train_accuracy / train_count
    train_loss = train_loss / train_count
    
    train_accuracy_list.append(train_accuracy)
    train_loss_list.append(train_loss)
    #         Evaluation on testing dataset
    model.eval()
    valid_accuracy=0.0
    
    pred = []
    lab = []
    
    
    for i, (images,labels) in enumerate(tqdm(valid_dataloader)):
        if torch.cuda.is_available():
            images=Variable(images.to(device))
            labels=Variable(labels.to(device))
        images = images.unsqueeze(1)
        images = images.float()
        outputs=model(images)
        _,prediction=torch.max(outputs.data,1)
        valid_accuracy+=int(torch.sum(prediction==labels.data))

        pred.extend(prediction.tolist())
        lab.extend(labels.tolist())
    
    valid_accuracy=valid_accuracy/valid_count
    valid_accuracy_list.append(valid_accuracy)
    
    if(max_acc<valid_accuracy):
        pred_labels=[]
        act_labels=[]
        pred_labels = pred
        act_labels = lab
        max_acc =valid_accuracy
        torch.save(model, 'model.pth')
    
    print('Epoch : '+str(epoch+1)+'/'+str(num_epochs)+'   Train Loss : '+str(train_loss)+'   Train Accuracy : '+str(train_accuracy)+'   Valid Accuracy : '+str(valid_accuracy))
#     print('Epoch:', epoch+1, '/', num_epochs, 'Train Loss:', train_loss.item(), 'Train Accuracy:', train_accuracy)
    
print('Finished Training')
print('max accuracy', max_acc)


In [ ]:
plt.plot(train_accuracy_list, label='Train Accuracy')
plt.plot(valid_accuracy_list, label='Valid Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Train and Test Accuracy')
plt.legend()

plt.show()

In [ ]:
# Calculate the confusion matrix
import seaborn as sns
conf_mat = confusion_matrix(act_labels, pred_labels)
# Plot confusion matrix heat map
sns.heatmap(conf_mat, cmap="flare",annot=True, fmt = "g",
            cbar_kws={"label":"color bar"},
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
# plt.savefig("ConfusionMatrix_BiLSTM.png")
plt.show()
from sklearn.metrics import f1_score
f1_score = f1_score(pred_labels, act_labels, average='weighted')
print('F1 Score : ', f1_score)

In [ ]:
import numpy as np
import sklearn.metrics

"""
Python compute equal error rate (eer)
ONLY tested on binary classification

:param label: ground-truth label, should be a 1-d list or np.array, each element represents the ground-truth label of one sample
:param pred: model prediction, should be a 1-d list or np.array, each element represents the model prediction of one sample
:param positive_label: the class that is viewed as positive class when computing EER
:return: equal error rate (EER)
"""
def compute_eer(act_labels, pred_labels, positive_label=1):
    # all fpr, tpr, fnr, fnr, threshold are lists (in the format of np.array)
    fpr, tpr, threshold = sklearn.metrics.roc_curve(act_labels, pred_labels)
    fnr = 1 - tpr

    # the threshold of fnr == fpr
    eer_threshold = threshold[np.nanargmin(np.absolute((fnr - fpr)))]

    # theoretically eer from fpr and eer from fnr should be identical but they can be slightly differ in reality
    eer_1 = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    eer_2 = fnr[np.nanargmin(np.absolute((fnr - fpr)))]

    # return the mean of eer from fpr and from fnr
    eer = (eer_1 + eer_2) / 2
    return eer

# Call the function to compute EER
eer_result = compute_eer(act_labels, pred_labels)

# Print the result
print("Equal Error Rate:", eer_result)

In [ ]:
from sklearn.metrics import f1_score, jaccard_score, matthews_corrcoef, hamming_loss,accuracy_score

f1 = f1_score(act_labels, pred_labels, average='weighted')
jaccard = jaccard_score(act_labels, pred_labels, average='weighted')
mcc = matthews_corrcoef(act_labels, pred_labels)
hloss = hamming_loss(act_labels, pred_labels)

print(f1)
print(jaccard)
print(mcc)
print(hloss)

In [ ]:
# create a list to store the predictions
# test_predictions_class = []
# test_labels = []

# # iterate through the test_dataset
# with torch.no_grad():
#     for inputs, actual_labels in test_dataloader:
#         if torch.cuda.is_available():
#             inputs=Variable(inputs.cuda())
#         # print(inputs)
#         outputs=model(inputs)
#         _,predicted=torch.max(outputs.data,1)
#         test_labels.extend(actual_labels.tolist())
#         test_predictions_class.extend(predicted.tolist())

#### Confusion Matrix

In [ ]:
# from sklearn.metrics import confusion_matrix
# import seaborn as sns

# cf_matrix = confusion_matrix(act_labels, pred_labels)

# sns.heatmap(cf_matrix, annot=False)

In [ ]:
# labels = test_dataset.class_to_idx
# confusion_matrix = torch.zeros(len(labels), len(labels))

# # Iterate through the data and compute the predictions and true labels
# for i in actual_labels:
#     confusion_matrix[actual_labels, pred_labels] += 1

# # Print the confusion matrix
# print(confusion_matrix)

# tuples = list(zip(actual_labels, pred_labels))

# # Define a dictionary to store the frequency of each tuple
# frequency_dict = {}

# # Iterate through the list of tuples and count the frequency of each non-equal tuple
# for t in tuples:
#     if t[0] != t[1]:
#         if t in frequency_dict:
#             frequency_dict[t] += 1
#         else:
#             frequency_dict[t] = 1

# # Sort the dictionary by frequency in descending order and get the top ten tuples
# number_of_top_freq = 50
# top_ten_tuples = sorted(frequency_dict.items(), key=lambda x: x[1], reverse=True)[:number_of_top_freq]

# # Print the top ten tuples and their frequencies
# for t, freq in top_ten_tuples:
#     print(f'Tuple: {t}, Actual Class: {test_dataset.classes[t[0]]} Predicted Class: {test_dataset.classes[t[1]]}, Frequency: {freq}')

In [ ]:
# print(pred_labels)

In [ ]:
# torch.save(model_whisp, r"/home/speechlab/Desktop/Siddharth/ASRUU/ModelsResults/Base/Run1_ep63_lr_03_01_003_drp_02.pt")

In [ ]:
# def pad(input_tensor):
#     if(input_tensor.shape[1]<400):
#         desired_dimensions = (1, 400, 768)
#         padding_amounts = (0,0,0,desired_dimensions[1] - input_tensor.shape[1])
#         padded_tensor = F.pad(input_tensor, padding_amounts, mode='constant',value = 0)
#     else:
#         padded_tensor = input_tensor[-1, 0:400 , :]
#     return padded_tensor

In [ ]:
# subfolders = [f.path for f in os.scandir(train_feat_save_path) if f.is_dir()]
# i=0
# mx=[]
# padded_data = []
# for subfolder in subfolders:
#     i=i+1
#     subfolder_name = os.path.basename(subfolder)
#     subfolder_path = os.path.join(train_feat_save_path, subfolder_name)

#     for file in os.listdir(subfolder_path):
      
#         file_path = os.path.join(subfolder_path, file)
#         data = torch.load(file_path, map_location='cpu').detach()
#         mx.append(data.shape[1])
#     #     data = data[-1, 0:400 , :]
#         print(data.shape)
#         break
#     break   


In [ ]:
# x=np.array(mx)
# print(len(np.unique(x)))
# print(len(x))

In [ ]:
# # plt.hist(mx, bins=361)
# plt.show()

In [ ]:
# import torch
# import torch.nn.functional as F

# # Assuming you have a tensor with dimensions [1, 137, 768]
# input_tensor = torch.randn(1, 137, 768)

# # Define the desired output dimensions
# desired_dimensions = (1, 400, 768)

# # Compute the padding amounts for each dimension
# # padding_amounts = [(0, desired_dimensions[i + 1] - input_tensor.shape[i + 1]) for i in range(len(desired_dimensions) - 1)]
# padding_amounts = (0,0,0,desired_dimensions[1] - input_tensor.shape[1])
# # Pad the tensor
# padded_tensor = F.pad(input_tensor, padding_amounts, mode='constant',value = 0)

# # Verify the new dimensions
# print(padded_tensor.shape)
